In [122]:
from dataclasses import dataclass, field
from datetime import date, datetime
from typing import Iterator, Optional
from icalendar import Calendar

In [123]:
MIN_GRADE = 1
MAX_GRADE = 5
LOW_GRADES = {1, 2}
SECONDS_IN_HOUR = 3600
DEFAULT_TOP_COUNT = 3

# OOP

**Fred Brosman**

*13.04.2026*

## Ülesanne 1. Üliõpilaste hindamissüsteem

Selles ülesandes loon süsteemi, millega saab hallata üliõpilasi, nende aineid ja hindeid.

Lahenduses kasutan:
- klassi `Student`, mis hoiab ühe üliõpilase andmeid
- klassi `GradeSystem`, mis haldab kõiki üliõpilasi
- eraldi meetodeid hinnete lisamiseks, keskmise arvutamiseks ja tulemuste analüüsimiseks

Programm peab täitma järgmised nõuded:
1. leidma 3 parimat tudengit keskmise hinde põhjal
2. arvutama iga aine keskmise hinde ja sorteerima ained parimast alates
3. leidma tudengid, kellel on vähemalt ühes aines hinne 1 või 2

## 1.1 Klass Student

See klass kirjeldab ühte üliõpilast.
Igal üliõpilasel on ID, nimi ja hinnete sõnastik.

In [124]:
@dataclass
class Student:
    """
    Klass ühe üliõpilase andmete hoidmiseks.

    Attributes
    ----------
    id : int
        Üliõpilase ID.
    name : str
        Üliõpilase nimi.
    grades : dict
        Sõnastik, kus võtmed on ainete nimed ja väärtused on hinded.
    """

    id: int
    name: str
    grades: dict[str, int] = field(default_factory=dict)

    def add_grade(self, course: str, grade: int) -> None:
        """
        Lisab üliõpilasele hinde kindlas aines.

        Parameters
        ----------
        course : str
            Aine nimi.
        grade : int
            Hinne vahemikus 1 kuni 5.
        """
        if not course or not course.strip():
            raise ValueError("Aine nimi ei tohi olla tühi.")
        if grade < MIN_GRADE or grade > MAX_GRADE:
            raise ValueError(f"Hinne peab olema vahemikus {MIN_GRADE} kuni {MAX_GRADE}.")
        self.grades[course.strip()] = grade

    def get_average(self) -> float:
        """
        Arvutab üliõpilase kõikide hinnete keskmise.

        Returns
        -------
        float
            Keskmine hinne. Kui hindeid ei ole, tagastatakse 0.
        """
        if not self.grades:
            return 0.0
        return sum(self.grades.values()) / len(self.grades)

## 1.2 Klass GradeSystem

See klass haldab kõiki üliõpilasi ja võimaldab teha ülesandes nõutud analüüse.

In [125]:
@dataclass
class GradeSystem:
    """
    Klass üliõpilaste hindamissüsteemi haldamiseks.

    Attributes
    ----------
    students : list
        Nimekiri Student tüüpi objektidest.
    """

    students: list[Student] = field(default_factory=list)

    def add_student(self, student: Student) -> None:
        """
        Lisab süsteemi ühe üliõpilase.

        Parameters
        ----------
        student : Student
            Süsteemi lisatav üliõpilane.
        """
        self.students.append(student)

    def get_top_students(self, count: int = DEFAULT_TOP_COUNT) -> list[tuple[str, int, float]]:
        """
        Leiab parimad tudengid keskmise hinde põhjal.

        Parameters
        ----------
        count : int
            Mitu tudengit tagastatakse.

        Returns
        -------
        list
            Nimekiri tuplena kujul (nimi, id, keskmine_hinne).
        """

        if count < 0:
            raise ValueError("count ei tohi olla negatiivne.")

        sorted_students = sorted(
            self.students,
            key=lambda student: student.get_average(),
            reverse=True
        )

        result = []
        for student in sorted_students[:count]:
            result.append((student.name, student.id, round(student.get_average(), 2)))
        return result

    def get_course_averages(self) -> list[tuple[str, float]]:
        """
        Arvutab iga aine keskmise hinde.

        Returns
        -------
        list
            Nimekiri tuplena kujul (aine, keskmine_hinne),
            sordituna parimast ainest alates.
        """
        course_grades: dict[str, list[int]] = {}

        for student in self.students:
            for course, grade in student.grades.items():
                if course not in course_grades:
                    course_grades[course] = []
                course_grades[course].append(grade)

        averages = []
        for course, grades in course_grades.items():
            average = sum(grades) / len(grades)
            averages.append((course, round(average, 2)))

        return sorted(averages, key=lambda item: item[1], reverse=True)

    def get_students_with_low_grades(self) -> list[tuple[str, list[str]]]:
        """
        Leiab tudengid, kellel on vähemalt ühes aines hinne 1 või 2.

        Returns
        -------
        list
            Nimekiri tuplena kujul (nimi, [ained]).
        """
        result = []

        for student in self.students:
            low_courses = []

            for course, grade in student.grades.items():
                if grade in LOW_GRADES:
                    low_courses.append(course)

            if low_courses:
                result.append((student.name, low_courses))

        return result

## 1.3 Abifunktsioonid tulemuste väljastamiseks

Järgmised funktsioonid aitavad väljastada tulemusi selgemal kujul.

In [126]:
def prindi_parimad_tudengid(parimad: list[tuple[str, int, float]]) -> None:
    """
    Väljastab 3 parimat tudengit.

    Parameters
    ----------
    parimad : list
        Nimekiri kujul (nimi, id, keskmine_hinne).
    """
    print("3 parimat tudengit:")
    for nimi, tudengi_id, keskmine in parimad:
        print(f"Nimi: {nimi}, ID: {tudengi_id}, keskmine hinne: {keskmine}")


def prindi_aine_keskmised(aine_keskmised: list[tuple[str, float]]) -> None:
    """
    Väljastab ainete keskmised hinded.

    Parameters
    ----------
    aine_keskmised : list
        Nimekiri kujul (aine, keskmine_hinne).
    """
    print("\nAinete keskmised hinded:")
    for aine, keskmine in aine_keskmised:
        print(f"{aine}: {keskmine}")


def prindi_madalad_hinded(madalad_hinded: list[tuple[str, list[str]]]) -> None:
    """
    Väljastab tudengid, kellel esineb madal hinne.

    Parameters
    ----------
    madalad_hinded : list
        Nimekiri kujul (nimi, [ained]).
    """
    print("\nTudengid, kellel on vähemalt ühes aines hinne 1 või 2:")
    for nimi, ained in madalad_hinded:
        print(f"{nimi}: {', '.join(ained)}")

## 1.4 Testandmete loomine

Selles plokis loon mõned näidisüliõpilased ja lisan neile hinded.

In [127]:
student1 = Student(1, "Mari")
student1.add_grade("Matemaatika", 5)
student1.add_grade("Programmeerimine", 4)
student1.add_grade("Füüsika", 5)

student2 = Student(2, "Juhan")
student2.add_grade("Matemaatika", 3)
student2.add_grade("Programmeerimine", 2)
student2.add_grade("Füüsika", 4)

student3 = Student(3, "Kati")
student3.add_grade("Matemaatika", 5)
student3.add_grade("Programmeerimine", 5)
student3.add_grade("Füüsika", 4)

student4 = Student(4, "Rasmus")
student4.add_grade("Matemaatika", 1)
student4.add_grade("Programmeerimine", 3)
student4.add_grade("Füüsika", 2)

student5 = Student(5, "Liis")
student5.add_grade("Matemaatika", 4)
student5.add_grade("Programmeerimine", 5)
student5.add_grade("Füüsika", 5)

## 1.5 Hindamissüsteemi loomine

Järgmises plokis lisan kõik loodud tudengid süsteemi.

In [128]:
system = GradeSystem()

system.add_student(student1)
system.add_student(student2)
system.add_student(student3)
system.add_student(student4)
system.add_student(student5)

## 1.6 Tulemuste arvutamine ja väljastamine

Selles plokis käivitan kõik ülesandes nõutud funktsionaalsused ja väljastan tulemused.

In [129]:
parimad = system.get_top_students()
aine_keskmised = system.get_course_averages()
madalad_hinded = system.get_students_with_low_grades()

prindi_parimad_tudengid(parimad)
prindi_aine_keskmised(aine_keskmised)
prindi_madalad_hinded(madalad_hinded)

3 parimat tudengit:
Nimi: Mari, ID: 1, keskmine hinne: 4.67
Nimi: Kati, ID: 3, keskmine hinne: 4.67
Nimi: Liis, ID: 5, keskmine hinne: 4.67

Ainete keskmised hinded:
Füüsika: 4.0
Programmeerimine: 3.8
Matemaatika: 3.6

Tudengid, kellel on vähemalt ühes aines hinne 1 või 2:
Juhan: Programmeerimine
Rasmus: Matemaatika, Füüsika


## Ülesanne 2. iCalendar tunniplaani analüüs OOP-iga

Selles ülesandes loen tunniplaani andmed `.ics` failist ja salvestan need
klasside eksemplaridesse.

Kasutan kolme põhiklassi:
- `Event` – üks sündmus
- `Course` – üks õppeaine koos sündmustega
- `Timetable` – kogu tunniplaan

Lahenduses kasutan objektorienteeritud programmeerimist, kapseldamist,
valideerimist, tüübimärgendusi ja docstring'e.

## 2.1 Klass Event

See klass kirjeldab ühte `.ics` faili sündmust.
Siin kontrollin ka, et sündmuse nimi ei oleks tühi ja lõpuaeg oleks algusajast hilisem.

In [130]:
@dataclass
class Event:
    """
    Klass ühe tunniplaani sündmuse hoidmiseks.
    """

    dtstart: datetime
    dtend: datetime
    location: Optional[str] = None
    description: Optional[str] = None
    __summary: str = field(init=False, repr=False)

    def __init__(
        self,
        summary: str,
        dtstart: datetime,
        dtend: datetime,
        location: Optional[str] = None,
        description: Optional[str] = None
    ) -> None:
        """
        Loob uue sündmuse.
        """
        if not summary or not summary.strip():
            raise ValueError("Sündmuse nimi ei tohi olla tühi.")

        if dtend <= dtstart:
            raise ValueError("Sündmuse lõpuaeg peab olema algusajast hilisem.")

        self.__summary = summary.strip()
        self.dtstart = dtstart
        self.dtend = dtend
        self.location = location.strip() if location and location.strip() else None
        self.description = description.strip() if description and description.strip() else None

    @property
    def summary(self) -> str:
        """
        Tagastab sündmuse nime.
        """
        return self.__summary

    def duration_hours(self) -> float:
        """
        Arvutab sündmuse kestuse tundides.
        """
        return (self.dtend - self.dtstart).total_seconds() / SECONDS_IN_HOUR

    def __str__(self) -> str:
        """
        Tagastab sündmuse loetava tekstiesituse.
        """
        koht = self.location if self.location else "Asukoht puudub"
        return (
            f"{self.summary} | "
            f"{self.dtstart.strftime('%d.%m.%Y %H:%M')} - "
            f"{self.dtend.strftime('%H:%M')} | "
            f"{koht}"
        )

## 2.2 Klass Course

See klass koondab ühe õppeaine kõik sündmused.

In [131]:
@dataclass
class Course:
    """
    Klass ühe õppeaine andmete hoidmiseks.

    Attributes
    ----------
    code : str
        Õppeaine kood.
    teacher : str
        Õppejõu nimi.
    events : list[Event]
        Selle ainega seotud sündmused.
    """

    code: str
    teacher: str
    events: list[Event] = field(default_factory=list)
    __name: str = field(init=False, repr=False)

    def __init__(
        self,
        code: str,
        name: str,
        teacher: str,
        events: Optional[list[Event]] = None
    ) -> None:
        """
        Loob uue õppeaine.

        Parameters
        ----------
        code : str
            Õppeaine kood.
        name : str
            Õppeaine nimi.
        teacher : str
            Õppejõu nimi.
        events : Optional[list[Event]]
            Õppeaine sündmuste nimekiri.

        Raises
        ------
        ValueError
            Kui aine nimi on tühi.
        """
        if not name or not name.strip():
            raise ValueError("Õppeaine nimi ei tohi olla tühi.")

        self.code = code.strip() if code and code.strip() else "TUNDMATU"
        self.__name = name.strip()
        self.teacher = teacher.strip() if teacher and teacher.strip() else "Tundmatu"
        self.events = events[:] if events else []

    @property
    def name(self) -> str:
        """
        Tagastab õppeaine nime.

        Returns
        -------
        str
            Õppeaine nimi.
        """
        return self.__name

    def add_event(self, event: Event) -> None:
        """
        Lisab ainele ühe sündmuse.

        Parameters
        ----------
        event : Event
            Lisatav sündmus.
        """
        self.events.append(event)

    def event_count(self) -> int:
        """
        Tagastab aine sündmuste arvu.

        Returns
        -------
        int
            Sündmuste arv.
        """
        return len(self.events)

    def total_hours(self) -> float:
        """
        Arvutab aine kõigi sündmuste kogukestuse tundides.

        Returns
        -------
        float
            Kogukestus tundides.
        """
        return sum(event.duration_hours() for event in self.events)

    def __str__(self) -> str:
        """
        Tagastab õppeaine loetava tekstiesituse.

        Returns
        -------
        str
            Õppeaine tekstina.
        """
        return (
            f"{self.code} - {self.name} | "
            f"Õppejõud: {self.teacher} | "
            f"Sündmusi: {len(self.events)}"
        )

## 2.3 Klass Timetable

See klass hoiab kogu tunniplaani ja sisaldab meetodit `.ics` faili lugemiseks.

In [132]:
@dataclass
class Timetable:
    """
    Klass kogu tunniplaani hoidmiseks ja analüüsimiseks.

    Attributes
    ----------
    courses : list[Course]
        Kõik tunniplaanis olevad õppeained.
    """

    courses: list["Course"] = field(default_factory=list)

    def __str__(self) -> str:
        """
        Tagastab tunniplaani lühikirjelduse.

        Returns
        -------
        str
            Tunniplaani kirjeldus.
        """
        return (
            f"Tunniplaan: {len(self.courses)} õppeainet, "
            f"{self.event_count()} sündmust"
        )

    def __iter__(self) -> Iterator["Event"]:
        """
        Võimaldab käia kõik sündmused for-tsükliga läbi.

        Yields
        ------
        Event
            Järgmine sündmus tunniplaanis.
        """
        for course in self.courses:
            for event in course.events:
                yield event

    def add_course(self, course: Course) -> None:
        """
        Lisab tunniplaani ühe õppeaine.

        Parameters
        ----------
        course : Course
            Lisatav õppeaine.
        """
        self.courses.append(course)

    def all_events(self) -> list["Event"]:
        """
        Tagastab kõik sündmused ühe nimekirjana, algusaja järgi sorditult.
        """
        events = []
        for course in self.courses:
            events.extend(course.events)

        return sorted(events, key=lambda event: event.dtstart)

    def event_count(self) -> int:
        """
        Tagastab kõigi sündmuste koguarvu.

        Returns
        -------
        int
            Sündmuste arv.
        """
        return sum(course.event_count() for course in self.courses)

    def total_hours(self) -> float:
        """
        Arvutab kogu tunniplaani tundide koguarvu.

        Returns
        -------
        float
            Kõigi sündmuste kogukestus tundides.
        """
        return sum(event.duration_hours() for event in self.all_events())

    def meetings_per_course(self) -> dict[str, int]:
        """
        Tagastab, mitu kohtumist on igas aines.

        Returns
        -------
        dict[str, int]
            Sõnastik kujul {aine_nimi: kohtumiste_arv}.
        """
        tulemus = {}

        for course in self.courses:
            if course.name not in tulemus:
                tulemus[course.name] = 0

            tulemus[course.name] += course.event_count()

        return tulemus

    def courses_by_teacher(self) -> dict[str, list[str]]:
        """
        Tagastab ainete nimekirja õppejõu järgi.

        Returns
        -------
        dict[str, list[str]]
            Sõnastik kujul {õppejõud: [ained]}.
        """
        tulemus = {}

        for course in self.courses:
            if course.teacher not in tulemus:
                tulemus[course.teacher] = []

            tulemus[course.teacher].append(course.name)

        for teacher in tulemus:
            tulemus[teacher] = sorted(set(tulemus[teacher]))

        return tulemus

    def next_two_courses(self) -> list[str]:
        """
        Leiab kaks erinevat ainet, mida õpetatakse kõige lähiajal.

        Returns
        -------
        list[str]
            Kahe lähima aine nimekiri.
        """
        koik_sundmused = [
            (course.name, event)
            for course in self.courses
            for event in course.events
        ]

        if not koik_sundmused:
            return []

        koik_sundmused.sort(key=lambda item: item[1].dtstart)

        esimene_aeg = koik_sundmused[0][1].dtstart
        praegu = datetime.now(esimene_aeg.tzinfo) if esimene_aeg.tzinfo else datetime.now()

        ained = []
        for course_name, event in koik_sundmused:
            if event.dtstart >= praegu and course_name not in ained:
                ained.append(course_name)
            if len(ained) == 2:
                break

        return ained

    @staticmethod
    def ensure_datetime(value) -> datetime:
        """
        Teisendab .ics failist loetud väärtuse vajadusel datetime kujule.

        Parameters
        ----------
        value
            .ics failist loetud kuupäev või kuupäev-kellaaeg.

        Returns
        -------
        datetime
            Datetime kujul väärtus.
        """
        if isinstance(value, datetime):
            return value

        return datetime.combine(value, datetime.min.time())

    @staticmethod
    def extract_teacher(description: Optional[str]) -> str:
        """
        Proovib kirjeldusest leida õppejõu nime.

        Parameters
        ----------
        description : Optional[str]
            Sündmuse kirjeldus.

        Returns
        -------
        str
            Õppejõu nimi või 'Tundmatu'.
        """
        if not description:
            return "Tundmatu"

        for rida in description.splitlines():
            puhas = rida.strip()
            lower = puhas.lower()

            if lower.startswith("õppejõud:") or lower.startswith("teacher:"):
                osad = puhas.split(":", 1)
                if len(osad) == 2 and osad[1].strip():
                    return osad[1].strip()

        return "Tundmatu"

    @staticmethod
    def extract_course_info(summary: str, description: Optional[str]) -> tuple[str, str, str]:
        """
        Proovib summary ja description väljadest leida aine koodi, nime ja õppejõu.

        Parameters
        ----------
        summary : str
            Sündmuse nimi.
        description : Optional[str]
            Sündmuse kirjeldus.

        Returns
        -------
        tuple[str, str, str]
            (aine_kood, aine_nimi, õppejõud)
        """
        code = summary.strip()
        name = summary.strip()
        teacher = Timetable.extract_teacher(description)

        if " - " in summary:
            vasak, parem = summary.split(" - ", 1)
            if vasak.strip():
                code = vasak.strip()
            if parem.strip():
                name = parem.strip()

        return code, name, teacher

    @classmethod
    def from_ical(cls, filename: str) -> "Timetable":
        """
        Loeb tunniplaani .ics failist ja tagastab Timetable objekti.

        Parameters
        ----------
        filename : str
            Faili nimi või tee.

        Returns
        -------
        Timetable
            Täidetud tunniplaani objekt.
        """
        with open(filename, "rb") as file:
            cal = Calendar.from_ical(file.read())

        courses_by_key: dict[tuple[str, str, str], Course] = {}

        for component in cal.walk():
            if component.name != "VEVENT":
                continue

            dtstart_field = component.get("dtstart")
            dtend_field = component.get("dtend")

            if dtstart_field is None or dtend_field is None:
                continue

            summary = str(component.get("summary", "")).strip()
            if not summary:
                continue

            dtstart = cls.ensure_datetime(dtstart_field.dt)
            dtend = cls.ensure_datetime(dtend_field.dt)
            location = str(component.get("location")).strip() if component.get("location") else None
            description = str(component.get("description")).strip() if component.get("description") else None

            event = Event(
                summary=summary,
                dtstart=dtstart,
                dtend=dtend,
                location=location,
                description=description
            )

            code, name, teacher = cls.extract_course_info(summary, description)
            key = (code, name, teacher)

            if key not in courses_by_key:
                courses_by_key[key] = Course(
                    code=code,
                    name=name,
                    teacher=teacher,
                    events=[]
                )

            courses_by_key[key].add_event(event)

        return cls(list(courses_by_key.values()))

## 2.4 Tunniplaani laadimine

Selles plokis loen andmed `.ics` failist ja kontrollin, et klassid töötavad õigesti.

In [133]:
tunniplaan = Timetable.from_ical("tunniplaan.ics")

print(tunniplaan)
print()

for course in tunniplaan.courses:
    print(course)

Tunniplaan: 11 õppeainet, 51 sündmust

RAM0620 - Programmeerimine | Õppejõud: Maarika Virkunen | Sündmusi: 4
RAM0620 - Programmeerimine | Õppejõud: Natalja Ivleva | Sündmusi: 4
NTR0450 - Inseneriinformaatika | Õppejõud: Valeria Juštšenko | Sündmusi: 4
NTR0450 - Inseneriinformaatika | Õppejõud: Žanna Gratšjova | Sündmusi: 3
RAM0800 - Digitaalloogika ja -süsteemid | Õppejõud: Oleg Shvets | Sündmusi: 3
RAM0800 - Digitaalloogika ja -süsteemid | Õppejõud: Monika Jänis | Sündmusi: 6
RAE1100 - Rakendusfüüsika | Õppejõud: Maarika Virkunen | Sündmusi: 3
RAE1100 - Rakendusfüüsika | Õppejõud: Sergei Pavlov | Sündmusi: 5
RAM0810 - Operatsioonisüsteemide administreerimine | Õppejõud: Oleg Shvets | Sündmusi: 5
EVK0033 - Eesti keel III | Õppejõud: Valentina Limonova | Sündmusi: 8
EVR0320 - Rohetehnoloogiate baaskursus | Õppejõud: Dmitri Nikitin | Sündmusi: 6


## 2.5 Kõikide sündmuste kuvamine

Järgmises plokis väljastan kõik tunniplaanis olevad sündmused.

In [134]:
for event in tunniplaan.all_events():
    print(event)

RAE1100 - Rakendusfüüsika | 07.02.2026 07:00 - 08:30 | VK1-42
RAM0620 - Programmeerimine | 07.02.2026 08:45 - 10:15 | VK1-42
NTR0450 - Inseneriinformaatika | 07.02.2026 11:00 - 14:15 | VK1-40
EVR0320 - Rohetehnoloogiate baaskursus | 08.02.2026 07:00 - 10:15 | VK1-9
RAE1100 - Rakendusfüüsika | 08.02.2026 11:00 - 14:15 | VK1-47
EVK0033 - Eesti keel III | 15.02.2026 07:00 - 10:15 | VK1-34
RAM0810 - Operatsioonisüsteemide administreerimine | 21.02.2026 07:00 - 10:15 | VK1-26
RAM0800 - Digitaalloogika ja -süsteemid | 21.02.2026 11:00 - 12:30 | VK1-44
RAE1100 - Rakendusfüüsika | 21.02.2026 12:45 - 14:15 | VK1-40
EVR0320 - Rohetehnoloogiate baaskursus | 22.02.2026 07:00 - 10:15 | VK1-9
RAM0800 - Digitaalloogika ja -süsteemid | 22.02.2026 11:00 - 12:30 | VK1-47
RAM0620 - Programmeerimine | 22.02.2026 12:45 - 16:00 | VK1-28
EVK0033 - Eesti keel III | 28.02.2026 11:00 - 14:15 | VK1-34
EVK0033 - Eesti keel III | 01.03.2026 07:00 - 10:15 | VK1-34
RAM0800 - Digitaalloogika ja -süsteemid | 07.03.202

## 2.6 Statistikafunktsioonid

Järgmisena lisan klassi `Timetable` sisse samad analüüsid, mis olid eelmises andmestruktuuride ülesandes punktides a–d.

In [135]:
tunniplaan = Timetable.from_ical("tunniplaan.ics")

print(tunniplaan)

print("\na) Mitu kohtumist on igas aines:")
for aine, arv in tunniplaan.meetings_per_course().items():
    print(f"{aine}: {arv}")

print("\nb) Ainete nimekiri õppejõu järgi:")
for oppejoud, ained in tunniplaan.courses_by_teacher().items():
    print(f"{oppejoud}: {', '.join(ained)}")

print("\nc) Kaks lähiajal õpetatavat ainet:")
for aine in tunniplaan.next_two_courses():
    print(aine)

Tunniplaan: 11 õppeainet, 51 sündmust

a) Mitu kohtumist on igas aines:
Programmeerimine: 8
Inseneriinformaatika: 7
Digitaalloogika ja -süsteemid: 9
Rakendusfüüsika: 8
Operatsioonisüsteemide administreerimine: 5
Eesti keel III: 8
Rohetehnoloogiate baaskursus: 6

b) Ainete nimekiri õppejõu järgi:
Maarika Virkunen: Programmeerimine, Rakendusfüüsika
Natalja Ivleva: Programmeerimine
Valeria Juštšenko: Inseneriinformaatika
Žanna Gratšjova: Inseneriinformaatika
Oleg Shvets: Digitaalloogika ja -süsteemid, Operatsioonisüsteemide administreerimine
Monika Jänis: Digitaalloogika ja -süsteemid
Sergei Pavlov: Rakendusfüüsika
Valentina Limonova: Eesti keel III
Dmitri Nikitin: Rohetehnoloogiate baaskursus

c) Kaks lähiajal õpetatavat ainet:
Inseneriinformaatika
Programmeerimine


## 2.7 Kahe lahenduse võrdlus

Allpool võrdlen lühidalt andmestruktuuride lahendust ja OOP lahendust.

| Küsimus | Andmestruktuuride lahendus | OOP lahendus |
|---|---|---|
| Kus hoitakse andmeid? | `dict`, `list`, `tuple` tüüpi andmestruktuurides | Klasside eksemplarides (`Event`, `Course`, `Timetable`) |
| Mis juhtub, kui andmestruktuur muutub? | Tuleb muuta mitut kohta koodis, sest võtmed ja indeksid võivad muutuda | Muudatused saab enamasti koondada klassidesse ja property-meetoditesse |
| Kui lihtne on lisada uut funktsiooni? | Väikese ülesande puhul on lihtne, aga suuremaks minnes muutub loogika hajusaks | Uut funktsiooni on lihtsam lisada klassi meetodina, sest andmed ja käitumine on koos |
| Kumb on loetavam? | Väikese ülesande puhul võib olla lühem | Minu arvates on OOP loetavam, sest klassinimed näitavad kohe, mida andmed tähendavad |
| Millal kasutaksite kumba? | Kasutaksin väikese ja ühekordse ülesande puhul | Kasutaksin OOP-i siis, kui programm kasvab suuremaks ja vajab paremat struktuuri |

## Ülesanne 3. Tarkvara

Selles ülesandes loon põhiklassi `Software` ja sellest tuletatud klassid:

- `Freeware`
- `Shareware`
- `CommercialSoftware`

Lahenduses kasutan:
- pärilust
- kapseldamist (`property`)
- objektide listi
- otsingumeetodeid erinevate tunnuste järgi

## 3.1 Põhiklass Software

See klass sisaldab kõigile tarkvaratüüpidele ühiseid atribuute:
- nimetus
- tootja
- installimise kuupäev

In [136]:
class Software:
    """
    Põhiklass tarkvara kirjeldamiseks.
    """

    def __init__(self, name: str, producer: str, install_date: date) -> None:
        """
        Loob uue tarkvara objekti.

        Parameters
        ----------
        name : str
            Tarkvara nimetus.
        producer : str
            Tarkvara tootja.
        install_date : date
            Tarkvara installimise kuupäev.
        """
        self.name = name
        self.producer = producer
        self.install_date = install_date

    @property
    def name(self) -> str:
        """
        Tagastab tarkvara nimetuse.
        """
        return self.__name

    @name.setter
    def name(self, value: str) -> None:
        """
        Kontrollib ja seab tarkvara nimetuse.
        """
        if not value or not value.strip():
            raise ValueError("Tarkvara nimetus ei tohi olla tühi.")
        self.__name = value.strip()

    @property
    def producer(self) -> str:
        """
        Tagastab tarkvara tootja.
        """
        return self.__producer

    @producer.setter
    def producer(self, value: str) -> None:
        """
        Kontrollib ja seab tarkvara tootja.
        """
        if not value or not value.strip():
            raise ValueError("Tootja nimi ei tohi olla tühi.")
        self.__producer = value.strip()

    def is_usable_today(self) -> bool:
        """
        Kontrollib, kas tarkvara saab kasutada tänasel kuupäeval.
        """
        return date.today() >= self.install_date

    def __str__(self) -> str:
        """
        Tagastab tarkvara loetava tekstiesituse.
        """
        return (
            f"Tüüp: {self.__class__.__name__}, "
            f"Nimetus: {self.name}, "
            f"Tootja: {self.producer}, "
            f"Installimise kuupäev: {self.install_date}"
        )

## 3.2 Klass Freeware

See klass pärib põhiklassist `Software`.
Freeware tarkvaral ei ole hinda ega kasutamise tähtaega.

In [137]:
class Freeware(Software):
    """
    Klass tasuta tarkvara kirjeldamiseks.
    """
    pass

## 3.3 Klass TimeLimitedSoftware

See on vaheklass tarkvarale, millel on kasutamise tähtaeg.

Selles klassis on ühine loogika:
- `expiry_date`
- selle valideerimine
- meetod `is_usable_today()`

Nii väldin sama loogika dubleerimist klassides `Shareware` ja `CommercialSoftware`.

In [138]:
class TimeLimitedSoftware(Software):
    """
    Vaheklass tarkvarale, millel on kasutamise tähtaeg.

    """

    def __init__(self, name: str, producer: str, install_date: date, expiry_date: date) -> None:
        """
        Loob uue tähtajalise tarkvara objekti.

        Parameters
        ----------
        name : str
            Tarkvara nimetus.
        producer : str
            Tarkvara tootja.
        install_date : date
            Tarkvara installimise kuupäev.
        expiry_date : date
            Tarkvara kasutamise tähtaeg.
        """
        super().__init__(name, producer, install_date)
        self.expiry_date = expiry_date

    @property
    def expiry_date(self) -> date:
        """
        Tagastab kasutamise tähtaja.
        """
        return self.__expiry_date

    @expiry_date.setter
    def expiry_date(self, value: date) -> None:
        """
        Kontrollib ja seab kasutamise tähtaja.
        """
        if value < self.install_date:
            raise ValueError("Kasutamise tähtaeg ei tohi olla varasem kui installimise kuupäev.")
        self.__expiry_date = value

    def is_usable_today(self) -> bool:
        """
        Kontrollib, kas tarkvara saab kasutada tänasel kuupäeval.
        """
        today = date.today()
        return self.install_date <= today <= self.expiry_date
    
    def __str__(self) -> str:
        return f"{super().__str__()}, Kasutamise tähtaeg: {self.expiry_date}"

## 3.4 Klass Shareware

See klass pärib nüüd vaheklassist `TimeLimitedSoftware`.
Shareware ei lisa uut atribuuti, vaid kasutab sama tähtajaloogikat.

In [139]:
class Shareware(TimeLimitedSoftware):
    """
    Klass prooviversiooniga tarkvara kirjeldamiseks.
    """
    pass

## 3.5 Klass CommercialSoftware

See klass pärib vaheklassist `TimeLimitedSoftware` ja lisab hinnainfo.

In [140]:
class CommercialSoftware(TimeLimitedSoftware):
    """
    Klass tasulise tarkvara kirjeldamiseks.
    """

    def __init__(
        self,
        name: str,
        producer: str,
        price: float,
        install_date: date,
        expiry_date: date
    ) -> None:
        """
        Loob uue CommercialSoftware objekti.

        Parameters
        ----------
        name : str
            Tarkvara nimetus.
        producer : str
            Tarkvara tootja.
        price : float
            Tarkvara hind.
        install_date : date
            Tarkvara installimise kuupäev.
        expiry_date : date
            Tarkvara kasutamise tähtaeg.
        """
        super().__init__(name, producer, install_date, expiry_date)
        self.price = price

    @property
    def price(self) -> float:
        """
        Tagastab tarkvara hinna.
        """
        return self.__price

    @price.setter
    def price(self, value: float) -> None:
        """
        Kontrollib ja seab tarkvara hinna.
        """
        if value <= 0:
            raise ValueError("Hind peab olema suurem kui 0.")
        self.__price = value

    def __str__(self) -> str:
        """
        Tagastab CommercialSoftware objekti tekstiesituse.
        """
        return (
            f"{super().__str__()}, "
            f"Hind: {self.price} € "
        )

## 3.6 Objektide list

Selles plokis loon erinevad tarkvaraobjektid ja salvestan need ühte listi.

In [141]:
software_list = [
    Freeware("Visual Studio Code", "Microsoft", date(2025, 9, 1)),
    Freeware("Notepad++", "Notepad++ Team", date(2025, 8, 15)),
    Shareware("WinRAR", "RARLAB", date(2026, 3, 1), date(2026, 4, 30)),
    Shareware("Sublime Text", "Sublime HQ", date(2026, 2, 10), date(2026, 5, 10)),
    CommercialSoftware("Microsoft Office", "Microsoft", 99.99, date(2026, 1, 1), date(2027, 1, 1)),
    CommercialSoftware("Adobe Photoshop", "Adobe", 149.99, date(2025, 12, 1), date(2026, 12, 1))
]

## 3.7 Kõikide andmete kuvamine

Järgmises plokis väljastan kõik tarkvaraobjektid ekraanile.

In [142]:
for software in software_list:
    print(software)

Tüüp: Freeware, Nimetus: Visual Studio Code, Tootja: Microsoft, Installimise kuupäev: 2025-09-01
Tüüp: Freeware, Nimetus: Notepad++, Tootja: Notepad++ Team, Installimise kuupäev: 2025-08-15
Tüüp: Shareware, Nimetus: WinRAR, Tootja: RARLAB, Installimise kuupäev: 2026-03-01, Kasutamise tähtaeg: 2026-04-30
Tüüp: Shareware, Nimetus: Sublime Text, Tootja: Sublime HQ, Installimise kuupäev: 2026-02-10, Kasutamise tähtaeg: 2026-05-10
Tüüp: CommercialSoftware, Nimetus: Microsoft Office, Tootja: Microsoft, Installimise kuupäev: 2026-01-01, Kasutamise tähtaeg: 2027-01-01, Hind: 99.99 € 
Tüüp: CommercialSoftware, Nimetus: Adobe Photoshop, Tootja: Adobe, Installimise kuupäev: 2025-12-01, Kasutamise tähtaeg: 2026-12-01, Hind: 149.99 € 


## 3.8 Otsingumeetodid

Selles plokis loon funktsioonid tarkvara otsimiseks:
- tarkvara tüübi järgi
- nimetuse järgi
- tootja järgi
- tarkvara järgi, mida saab kasutada tänasel kuupäeval

In [143]:
def search_by_type(software_items: list[Software], software_type: type[Software]) -> list[Software]:
    """
    Otsib tarkvara tüübi järgi.

    Parameters
    ----------
    software_items : list
        Tarkvaraobjektide nimekiri.
    software_type : type
        Otsitava klassi tüüp.

    Returns
    -------
    list
        Leitud tarkvaraobjektid.
    """
    return [software for software in software_items if isinstance(software, software_type)]


def search_by_name(software_items: list[Software], keyword: str) -> list[Software]:
    """
    Otsib tarkvara nimetuse või selle osa järgi.

    Parameters
    ----------
    software_items : list
        Tarkvaraobjektide nimekiri.
    keyword : str
        Otsitav märksõna.

    Returns
    -------
    list
        Leitud tarkvaraobjektid.
    """
    keyword = keyword.strip().lower()
    if not keyword:
        return []
    return [software for software in software_items if keyword in software.name.lower()]



def search_by_producer(software_items: list[Software], producer: str) -> list[Software]:
    """
    Otsib tarkvara tootja järgi.

    Parameters
    ----------
    software_items : list
        Tarkvaraobjektide nimekiri.
    producer : str
        Otsitava tootja nimi.

    Returns
    -------
    list
        Leitud tarkvaraobjektid.
    """
    producer = producer.strip().lower()
    if not producer:
        return []
    return [software for software in software_items if producer in software.producer.lower()]


def search_usable_today(software_items: list[Software]) -> list[Software]:
    """
    Leiab tarkvara, mida saab kasutada tänasel kuupäeval.

    Parameters
    ----------
    software_items : list
        Tarkvaraobjektide nimekiri.

    Returns
    -------
    list
        Täna kasutatavad tarkvaraobjektid.
    """
    return [software for software in software_items if software.is_usable_today()]

## 3.9 Otsingute testimine

Järgmises plokis testin eelnevalt loodud otsingufunktsioone.

In [144]:
print("Freeware tarkvara:")
for software in search_by_type(software_list, Freeware):
    print(software)

print("\nNime järgi otsing: 'studio'")
for software in search_by_name(software_list, "studio"):
    print(software)

print("\nTootja järgi otsing: 'microsoft'")
for software in search_by_producer(software_list, "microsoft"):
    print(software)

print("\nTarkvara, mida saab kasutada täna:")
for software in search_usable_today(software_list):
    print(software)

Freeware tarkvara:
Tüüp: Freeware, Nimetus: Visual Studio Code, Tootja: Microsoft, Installimise kuupäev: 2025-09-01
Tüüp: Freeware, Nimetus: Notepad++, Tootja: Notepad++ Team, Installimise kuupäev: 2025-08-15

Nime järgi otsing: 'studio'
Tüüp: Freeware, Nimetus: Visual Studio Code, Tootja: Microsoft, Installimise kuupäev: 2025-09-01

Tootja järgi otsing: 'microsoft'
Tüüp: Freeware, Nimetus: Visual Studio Code, Tootja: Microsoft, Installimise kuupäev: 2025-09-01
Tüüp: CommercialSoftware, Nimetus: Microsoft Office, Tootja: Microsoft, Installimise kuupäev: 2026-01-01, Kasutamise tähtaeg: 2027-01-01, Hind: 99.99 € 

Tarkvara, mida saab kasutada täna:
Tüüp: Freeware, Nimetus: Visual Studio Code, Tootja: Microsoft, Installimise kuupäev: 2025-09-01
Tüüp: Freeware, Nimetus: Notepad++, Tootja: Notepad++ Team, Installimise kuupäev: 2025-08-15
Tüüp: Shareware, Nimetus: WinRAR, Tootja: RARLAB, Installimise kuupäev: 2026-03-01, Kasutamise tähtaeg: 2026-04-30
Tüüp: Shareware, Nimetus: Sublime Text, 

#  AI kasutamise vorm



**Üldinfo:**

Nimi: Fred Brosman

Ülesanne: OOP

Kuupäev: 16.04.2026

---

**1. Milliseid AI tööriistu kasutasid?**

ChatGPT, Copilot

---

**2. Too välja 2-3 näidet AI-prompt'idest, mida kasutasid:**

a) "Role: Tegutse Python objektorienteeritud programmeerimise juhendajana.
Task: Aita mul luua klassid Student ja GradeSystem, kus tudengile saab lisada hindeid, arvutada keskmise ning leida parimad tudengid ja madalate hinnetega tudengid.
Format: Näita lihtsat algajasõbralikku OOP-lahendust koos docstring'ide ja lühikeste selgitustega.
Goal: Soovin teha arusaadava üliõpilaste hindamissüsteemi."

b) "Role: Tegutse Python ja OOP juhendajana.
Task: Aita mul modelleerida .ics tunniplaani andmed klassidega Event, Course ja Timetable, lisada valideerimine, property ja statistikameetodid.
Format: Jaga lahendus väikesteks klassideks ja meetoditeks ning lisa docstring'id.
Goal: Soovin salvestada tunniplaani andmed klasside eksemplaridesse, mitte sõnastikesse."

c) "Role: Tegutse algaja Python õppija abistajana.
Task: Aita mul teha tarkvara klassihierarhia klassidega Software, Freeware, Shareware ja CommercialSoftware, kasutada pärilust, property't ja otsingufunktsioone.
Format: Näita loogilist OOP-lahendust koos näidisobjektide ja otsingunäidetega.
Goal: Soovin paremini mõista pärilust ja kapseldamist Pythonis."

---

**3. Millised koodiosad kirjutasid täielikult ise?**

- suure osa notebooki struktuurist ja markdown-pealkirjadest

- testandmete loomise üliõpilaste hindamissüsteemi jaoks

- tulemuste väljastamise abifunktsioonid, näiteks parimate tudengite, ainete keskmiste ja madalate hinnete kuvamine

- tunniplaani väljundi ja statistika kuvamise plokid notebookis

- tarkvaraobjektide näidislisti loomine

- mitmed nimede, kirjelduste ja markdown-plokkide sõnastused

---

**4. Milliseid AI poolt genereeritud lahendusi pidid parandama või ümber kirjutama?**

- kohandasin üliõpilaste hindamissüsteemi lahendust nii, et lisada sobivamad tüübimärgendused ja konstandid

- parandasin tunniplaani analüüsi loogikat, et sama aine kohtumised liidetaks õigesti kokku ka siis, kui sama ainet õpetavad erinevad õppejõud

- täiendasin .ics faili lugemise loogikat, et puuduvate väljade korral programm ei katkeks

- muutsin tarkvara klassihierarhiat nii, et korduvat tähtajaga seotud loogikat mitte dubleerida

- parandasin Software klassi nii, et valideerimine toimiks ka objekti loomisel läbi property setter'ite

- kohandasin mitmeid koodiosi loetavamaks ja notebooki jaoks sobivamaks

---

**5. Milliste probleemide lahendamisel aitas AI kõige rohkem?**

- OOP klasside vastutuste jaotamisel

- property, kapseldamise ja valideerimise loogika mõistmisel

- päriluse ja klassihierarhia puhastamisel tarkvara ülesandes

---

**6. Mida õppisid selle ülesande käigus tehniliselt?**

- kuidas modelleerida andmeid klasside ja objektidega

- kuidas kasutada @dataclass-i ja millal on mõistlik kirjutada __init__ ise

- kuidas kasutada property getter'eid ja setter'eid kapseldamise jaoks

- kuidas jagada loogika mitme klassi vahel nii, et igal klassil oleks oma vastutus

- kuidas kasutada pärilust ja vältida korduvat loogikat alamklassides

- Kuidas jaotada programm väiksemateks funktsioonideks, et kood oleks loetavam ja paremini testitav

---

**7. Enesehinnang:**

- [ ] **Vibe Coding** – kasutasin AI-d kiireks koodi genereerimiseks, ei muutnud palju
- [x] **Mixed** – kasutasin AI-d, aga kohandasin ja parandasin koodi
- [ ] **Deep Coding** – kasutasin AI-d abivahendina, mõistan täpselt, kuidas kood töötab

---

**8. Lühike põhjendus oma enesehinnangule (2-3 lauset):**

Kasutasin AI-d peamiselt abina, et paremini aru saada OOP ülesannete struktuurist ja keerulisematest osadest, näiteks kapseldamisest, .ics faili töötlemisest ja pärilusest. Samas muutsin saadud koodi päris palju ise, parandasid loogikavigu ja kohandasin lahendused oma notebooki jaoks sobivaks. Saan üldiselt aru, kuidas lahendus töötab, aga AI aitas mul asjad kiiremini ja kindlamalt kokku panna.